Imports

In [1]:
import sys
from pathlib import Path

# Add project root to path (go up 3 levels from current notebook location)
project_root = Path.cwd()
if 'DetectionModel' in str(project_root):
    # If we're inside DetectionModel folder, go up to MachineLearning root
    while project_root.name != 'MachineLearning' and project_root.parent != project_root:
        project_root = project_root.parent

sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from constants.detection.dataset_constants import RegModelConstants


To access the regression dataset we first need to read the csv containing the features

In [2]:
reg_df_path = Path(RegModelConstants.DATASET_METADATA_PATH)

reg_df = pd.read_csv(reg_df_path)

after reading the csv we need to filter the df to contain only relevant info to the regresion model

In [ ]:
features= reg_df.columns
bbox_features=list(filter(lambda feature:feature.startswith("bbox"),features))
centorid_features=list(filter(lambda feature:feature.startswith("centroid"),features))
volume_features=list(filter(lambda feature:feature.startswith("volume"),features))

print(f"Df columns before filtering: {features.tolist()}")

columns_to_drop = [
    RegModelConstants.FILE_NAME,
    RegModelConstants.PATIENT_ID,
    RegModelConstants.NOUDLE_INDEX,
    RegModelConstants.SLICE_INDEX,
    RegModelConstants.Features.FEATURE_ANNOTATION_COUNT,
    *bbox_features,
    *centorid_features,
    *volume_features,
    RegModelConstants.IMAGE_PATH,
    RegModelConstants.LABEL_PATH
]

reg_df.drop(columns=columns_to_drop,inplace=True)

print(f"Df columns after filtering: {reg_df.columns.tolist()}")

Df columns before filtering: Index(['filename', 'patient_id', 'split_group', 'nodule_index', 'slice_index',
       'diameter_mm', 'malignancy_score', 'spiculation', 'lobulation',
       'subtlety', 'sphericity', 'margin', 'texture', 'calcification',
       'internal_structure', 'annotation_count', 'centroid_Z', 'centroid_Y',
       'centroid_x', 'bbox_x', 'bbox_y', 'bbox_w', 'bbox_h', 'image_path',
       'label_path', 'volume_depth', 'volume_height', 'volume_width'],
      dtype='object')
Df columns after filtering: Index(['split_group', 'diameter_mm', 'malignancy_score', 'spiculation',
       'lobulation', 'subtlety', 'sphericity', 'margin', 'texture',
       'calcification', 'internal_structure'],
      dtype='object')


We'll get some basic info about the csv in order to plan how we need analyze it

In [4]:
def get_df_info(df:pd.DataFrame):
    print(f"dataset shape = {df.shape}")
    print(f"Number of Nodules data to anlayze= {df.shape[0]}")

print("-"*25)

train_df= reg_df[reg_df[RegModelConstants.SPLIT_GROUP]==RegModelConstants.TRAIN_SPLIT]
test_df= reg_df[reg_df[RegModelConstants.SPLIT_GROUP]==RegModelConstants.TEST_SPLIT]
val_df= reg_df[reg_df[RegModelConstants.SPLIT_GROUP]==RegModelConstants.VAL_SPLIT]

print("Train info")
get_df_info(train_df)
print("-"*25)
print("Test info")
get_df_info(test_df)
print("-"*25)
print("Train info")
get_df_info(val_df)



-------------------------
Train info
dataset shape = (5367, 11)
Number of Nodules data to anlayze= 5367
-------------------------
Test info
dataset shape = (1000, 11)
Number of Nodules data to anlayze= 1000
-------------------------
Train info
dataset shape = (1013, 11)
Number of Nodules data to anlayze= 1013


After getting some basic info we need to split the features into X and Y values in order to prepare it for the model

In [10]:


x_features = reg_df.drop(columns=[RegModelConstants.SPLIT_GROUP, f"{RegModelConstants.Features.FEATURE_MALIGNANCY}_score"])


y_features = reg_df[f"{RegModelConstants.Features.FEATURE_MALIGNANCY}_score"]

print(f"X shape: {x_features.shape}")
print(f"Y shape: {y_features.shape}")


X shape: (7380, 9)
Y shape: (7380,)
